# Trabajo Práctico: Laboratorio de Métodos Cuantitativos Aplicados a la Gestión
**Materia:** Laboratorio de Métodos Cuantitativos Aplicados a la Gestión (FCE UBA)  
**Dataset:** DataCo Smart Supply Chain  

Esta notebook unificada integra todo el trabajo del grupo: la exploración inicial, la limpieza y validación de datos a nivel de pedido, el análisis estadístico descriptivo y comparativo, y el análisis gráfico avanzado con sus respectivas interpretaciones económicas.

# SECCIÓN 1 — CARGA, EXPLORACIÓN E INVENTARIO INICIAL DE DATOS

En esta primera sección se realiza la importación de las librerías necesarias para el análisis y la carga remota del dataset oficial desde el repositorio de GitHub. A continuación, se ejecuta un primer control exploratorio para auditar la estructura general, las dimensiones del dataset, la cobertura temporal y geográfica, así como la presencia de valores faltantes y filas duplicadas antes de proceder con las transformaciones de datos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización para claridad en decimales y columnas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# Carga remota del dataset desde la URL raw de GitHub con encoding latin-1
url_raw = "https://raw.githubusercontent.com/JuanchiWienberg/TP---LAB.-DE-MET.-CUANT.-APLICADOS-A-LA-GEST./main/DataCoSupplyChainDataset.csv"
df = pd.read_csv(url_raw, encoding="latin-1")

print("Materia prima cargada con éxito. Filas originales:", df.shape[0])
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas\n")

print("--- PRIMERAS 5 FILAS (HEAD) ---")
display(df.head())

print("--- INFORMACIÓN DE COLUMNAS Y TIPOS DE DATOS ---")
df.info()

# Cobertura temporal
if "order date (DateOrders)" in df.columns:
    df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
    print(f"\nPeríodo temporal cubierto: {df['order date (DateOrders)'].min()} a {df['order date (DateOrders)'].max()}")

# Cobertura geográfica
if "Order Region" in df.columns:
    print(f"Regiones comerciales representadas ({df['Order Region'].nunique()} en total):")
    print(df["Order Region"].unique())

# Evaluación de calidad inicial
nulos = df.isnull().sum()
print("\n--- COLUMNAS CON VALORES NULOS ---")
print(nulos[nulos > 0])

duplicados = df.duplicated().sum()
print(f"\nCantidad de filas totalmente duplicadas: {duplicados}")

print("\n--- ESTADÍSTICAS DESCRIPTIVAS INICIALES ---")
display(df.describe().T)


## Pregunta de Investigación Central

1. **¿Cuál es la pregunta?**  
   ¿En qué medida el riesgo de entregas tardías (`Late_delivery_risk`) afecta el margen de ganancia operativa en los diferentes segmentos de clientes y regiones comerciales de DataCo?

2. **¿Por qué vale la pena responderla?**  
   Permite identificar si las fallas en los tiempos logísticos están asociadas a variaciones en la rentabilidad operativa de la empresa de forma diferenciada según el mercado o el perfil del cliente.

---
# SECCIÓN 2 — Preparación y validación de los datos

Esta sección continúa el trabajo de exploración inicial. Revisa la unidad de análisis, justifica las transformaciones necesarias y deja una tabla con un registro por pedido para el análisis posterior.

## Etapa 0 — Punto de partida y decisiones pendientes
La exploración inicial registró 180.519 filas y 53 columnas, sin filas completamente duplicadas. Antes de comparar ganancias, debemos tener en cuenta que la base está organizada por artículos de pedido: un mismo `Order Id` puede aparecer en varias filas. Por eso, la tabla original no debe tratarse directamente como si cada fila fuera un pedido.

## Etapa 1 — Validar la unidad de análisis y las variables económicas

### 1.1. Comprobar cuántas filas tiene cada pedido
Contamos las filas asociadas a cada `Order Id`. Esto permite confirmar si la unidad de observación original coincide con la unidad que necesita nuestra pregunta de investigación.


In [ ]:
# 1.1. Comprobar cuántas filas tiene cada pedido
filas_por_pedido = df.groupby("Order Id").size()

print(f"Filas de la base original: {len(df):,}")
print(f"Pedidos únicos: {df['Order Id'].nunique():,}")
print(f"Pedidos con una sola fila: {(filas_por_pedido == 1).sum():,}")
print(f"Pedidos con varias filas: {(filas_por_pedido > 1).sum():,}")
print(f"Máximo de filas para un pedido: {filas_por_pedido.max()}")

# 1.2. Revisar un pedido con varias filas
id_ejemplo = filas_por_pedido[filas_por_pedido > 1].index[0]

columnas_ejemplo = [
    "Order Id",
    "Order Item Id",
    "Order Item Quantity",
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Delivery Status"
]

display(
    df.loc[df["Order Id"] == id_ejemplo, columnas_ejemplo]
    .sort_values("Order Item Id")
)

# 1.3. Comprobar qué variables se mantienen constantes dentro de cada pedido
variables_constantes = [
    "Late_delivery_risk",
    "Delivery Status",
    "Customer Segment",
    "Order Region"
]

print("¿Coinciden Order Profit Per Order y Benefit per order en todas las filas?")
print(df["Order Profit Per Order"].eq(df["Benefit per order"]).all())

print("\nCantidad máxima de valores distintos por pedido:")
display(
    df.groupby("Order Id")[variables_constantes]
    .nunique(dropna=False)
    .max()
    .to_frame("Máximo de valores distintos")
)

print("\n¿Order Item Id identifica cada fila?")
print(f"Valores únicos de Order Item Id: {df['Order Item Id'].nunique():,}")
print(f"Filas de la base: {len(df):,}")


### 1.4. Revisar valores negativos y posibles outliers
Antes de construir una tabla por pedido, revisamos los valores negativos de las variables económicas y detectamos posibles valores atípicos mediante el rango intercuartílico (IQR).


In [ ]:
columnas_economicas = [
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Sales per customer"
]

resumen_negativos = []
for columna in columnas_economicas:
    valores = pd.to_numeric(df[columna], errors="coerce")
    resumen_negativos.append({
        "variable": columna,
        "negativos": int((valores < 0).sum()),
        "porcentaje_negativo": float((valores < 0).mean() * 100),
        "mínimo": valores.min(),
        "máximo": valores.max()
    })

display(pd.DataFrame(resumen_negativos).round(4))


**Decisión metodológica:** no se eliminan ganancias negativas ni posibles outliers por el solo hecho de ser negativos o extremos. La variable `Order Profit Per Order` se agrega por `Order Id` como suma del pedido, y el análisis conservará las observaciones para evaluar la variabilidad completa.

## Etapa 2 — Preparar las variables necesarias
Trabajamos sobre una copia para conservar intacto el DataFrame original. Normalizamos los espacios sobrantes en `Order Region` y aseguramos que las fechas tengan formato adecuado.


In [ ]:
df_base_preparada = df.copy()

# Normalizar espacios sobrantes en la región
df_base_preparada["Order Region"] = df_base_preparada["Order Region"].astype("string").str.strip()

# Convertir fechas
df_base_preparada["order date (DateOrders)"] = pd.to_datetime(
    df_base_preparada["order date (DateOrders)"], errors="coerce"
)
df_base_preparada["shipping date (DateOrders)"] = pd.to_datetime(
    df_base_preparada["shipping date (DateOrders)"], errors="coerce"
)


## Etapa 3 — Construir una tabla con un registro por pedido
Agrupamos las filas por `Order Id`. Sumamos los importes de ganancia registrados y `Order Item Total` para construir medidas agregadas por pedido, y conservamos el riesgo, el estado de entrega, el segmento y la región tomando el primer valor de cada grupo.


In [ ]:
variables_esperadas_constantes = [
    "Late_delivery_risk", "Delivery Status", "Customer Segment",
    "Order Region", "Order Status", "order date (DateOrders)"
]

df_pedidos = (
    df_base_preparada.groupby("Order Id")
    .agg(
        ganancia_total_pedido=("Order Profit Per Order", "sum"),
        ventas_totales_pedido=("Order Item Total", "sum"),
        cantidad_filas=("Order Item Id", "count"),
        riesgo_entrega_tardia=("Late_delivery_risk", "first"),
        estado_entrega=("Delivery Status", "first"),
        segmento_cliente=("Customer Segment", "first"),
        region=("Order Region", "first")
    )
    .reset_index()
)

# Margen de ganancia calculado a nivel pedido
df_pedidos["margen_ganancia_pedido"] = np.where(
    df_pedidos["ventas_totales_pedido"] != 0,
    df_pedidos["ganancia_total_pedido"] / df_pedidos["ventas_totales_pedido"],
    np.nan
)

print(f"Tabla de pedidos construida: {len(df_pedidos):,} filas y {df_pedidos.shape} columnas")
display(df_pedidos.head())


## Etapa 4 — Validar la tabla y dejarla lista para el análisis
Los pedidos con estado `Shipping canceled` se mantienen en `df_pedidos` para no perder información. Creamos por separado `df_pedidos_analisis`, que excluye esos pedidos de la comparación de entregas realizadas.


In [ ]:
variables_clave = [
    "Order Id",
    "ganancia_total_pedido",
    "ventas_totales_pedido",
    "riesgo_entrega_tardia",
    "estado_entrega",
    "segmento_cliente",
    "region"
]

print("¿Hay un único registro por pedido?")
print(df_pedidos["Order Id"].is_unique)

print("\nCantidad de pedidos por estado de entrega:")
display(df_pedidos["estado_entrega"].value_counts(dropna=False).to_frame("Cantidad de pedidos"))

df_pedidos_analisis = df_pedidos.loc[
    df_pedidos["estado_entrega"] != "Shipping canceled"
].copy()

print(f"\nPedidos disponibles para comparar entregas: {len(df_pedidos_analisis):,}")
display(df_pedidos_analisis[["ganancia_total_pedido", "ventas_totales_pedido", "margen_ganancia_pedido"]].describe().T)


---
# SECCIÓN 3 — Calcular estadísticas y comparar resultados (Integrante: Sebastián Morán)

En esta sección se desarrolla la parte estadístico-analítica del dataset para responder a la pregunta central. Calculamos las estadísticas descriptivas relevantes y construimos tablas agrupadas comparativas.


In [ ]:
# Etapa 1: Estadísticas descriptivas de las variables numéricas clave
variables_numericas = [
    "ventas_totales_pedido",
    "ganancia_total_pedido",
    "margen_ganancia_pedido"
]

estadisticas_descriptivas = df_pedidos_analisis[variables_numericas].describe().T
print("--- Estadísticas descriptivas relevantes ---")
display(estadisticas_descriptivas)

# Etapa 2: Utilizar groupby() para comparar los grupos por estado de entrega
comparación_entrega = (
    df_pedidos_analisis.groupby("estado_entrega")
    .agg(
       cantidad_pedidos=("Order Id", "count"),
       promedio_ventas=("ventas_totales_pedido", "mean"),
       promedio_ganancia=("ganancia_total_pedido", "mean"),
       margen_promedio=("margen_ganancia_pedido", "mean")
    )
   .reset_index()
)
print("\n--- Comparación por estado de entrega ---")
display(comparación_entrega)

# Etapa 3: Preparar las dos tablas de resultados agrupados
# Tabla 1: Segmento de cliente cruzado con el riesgo de entrega tardía
tabla_segmento_riesgo = (
    df_pedidos_analisis.groupby(["segmento_cliente", "riesgo_entrega_tardia"])
    .agg(
        total_pedidos=("Order Id", "count"),
        ganancia_total=("ganancia_total_pedido", "sum"),
        margen_promedio=("margen_ganancia_pedido", "mean")
    )
    .reset_index()
)
print("\n--- Tabla 1: Resultados agrupados por Segmento y Riesgo ---")
display(tabla_segmento_riesgo)

# Tabla 2: Resultados agrupados por Región de destino
tabla_region = (
    df_pedidos_analisis.groupby("region")
    .agg(
        total_pedidos=("Order Id", "count"),
        ventas_totales=("ventas_totales_pedido", "sum"),
        ganancia_promedio=("ganancia_total_pedido", "mean")
    )
    .reset_index()
    .sort_values(by="total_pedidos", ascending=False)
)
print("\n--- Tabla 2: Resultados agrupados por Región ---")
display(tabla_region)


## Etapa 4 — Interpretar las diferencias observadas (Análisis Descriptivo)

### 📝 Interpretación de la Tabla de Estado de Entrega (`comparación_entrega`)
Al analizar la tabla agrupada por estado de entrega, se observa que la gran mayoría de los envíos procesados corresponden a la categoría `Late delivery` (98.977 pedidos), seguidos por `Advance shipping` (41.592 pedidos) y `Shipping on time` (32.196 pedidos).
En términos de tendencia central, el volumen promedio de ventas por pedido se ubica entre $210,02 y $211,76 a través de los tres estados, mientras que la ganancia promedio por pedido oscila entre $24,67 y $25,18. En consecuencia, el **margen de ganancia promedio** se mantiene sumamente homogéneo entre categorías (11.83% para envíos anticipados, 11.64% para envíos a tiempo y 11.58% para envíos con entrega tardía). Descriptivamente, la métrica de margen promedio a nivel agregado no muestra variaciones significativas según la condición final de entrega del pedido.

### 📝 Interpretación de la Tabla 1 (Segmento de Cliente y Riesgo)
En la Tabla 1, al cruzar el segmento comercial con el indicador `Late_delivery_risk`, se observa que el margen promedio por pedido resulta prácticamente constante en los tres canales: `Consumer` (11.86% sin riesgo vs. 11.62% con riesgo), `Corporate` (12.43% sin riesgo vs. 11.69% con riesgo) y `Home Office` (12.39% sin riesgo vs. 11.47% con riesgo). La ganancia total acumulada responde de manera directa al volumen total de operaciones de cada segmento, siendo `Consumer` el de mayor volumen (41.488 pedidos totales), mientras que `Home Office` representa la menor escala de operaciones (14.283 pedidos).

### 📝 Interpretación de la Tabla 2 (Región de Destino)
En la Tabla 2, se constata la distribución geográfica del negocio. Regiones comerciales como `Western Europe` (14.180 pedidos, ventas acumuladas por $2.99M) y `Central America` (14.072 pedidos, ventas acumuladas por $2.98M) concentran los mayores volúmenes de transacción e ingresos brutos. Por su parte, la ganancia promedio por pedido registra variaciones entre territorios (por ejemplo, $29,04 en Western Europe frente a $18,52 en Southern Africa), lo que describe diferencias en la composición o valor promedio de las órdenes según el mercado geográfico.

## Etapa 5 — Evadir atribuir causalidad a relaciones que los datos solo permiten describir

**Nota metodológica:** Las agrupaciones y comparaciones realizadas mediante `groupby()` y estadísticas descriptivas muestran asociaciones y patrones observados en los datos históricos (por ejemplo, la coexistencia de variaciones en los márgenes regionales). Sin embargo, no implican una relación causal directa (no se puede afirmar categóricamente que el riesgo de entrega tardía cause por sí solo variaciones en la rentabilidad de cada pedido, ya que existen variables operativas no registradas en la base). Los datos son de naturaleza estrictamente descriptiva.

## Declaración de Uso de IA

Se utilizó la IA para asistir en la creación de las tablas agrupadas, la verificación de los cálculos y la redacción de las interpretaciones descriptivas.


# SECCIÓN 4 — Análisis Gráfico (Integrante 4: Luublue)

En esta sección se desarrolla el análisis visual y la exploración gráfica del dataset para responder a la pregunta central de investigación. Utilizando las variables previamente procesadas y normalizadas, nos enfocaremos en evaluar el comportamiento del indicador económico clave (`margen_ganancia_pedido`) en interacción con las alertas operativas (`riesgo_entrega_tardia`), abriendo el análisis según el segmento del cliente y la localización geográfica del mercado.

El estudio se estructurará mediante tres enfoques gráficos independientes: un análisis de distribución total (Boxplot), un estudio de brechas por canal comercial (Barras agrupadas) y una matriz de densidad territorial (Mapa de calor).

### Gráfico 1: Evaluación de la Distribución del Margen según Alerta Logística
**Objetivo metodológico:** Evaluar la distribución, la dispersión intercuartílica y la presencia de asimetrías o pérdidas en el margen operativo porcentual de los pedidos, contrastando el escenario normal (Sin Riesgo) contra el escenario ineficiente (Con Riesgo).


In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    x="riesgo_entrega_tardia",
    y="margen_ganancia_pedido",
    data=df_pedidos_analisis,
    palette="Set2",
    showfliers=True
)

plt.title("Distribución del Margen de Ganancia Operativa según Riesgo Logístico", fontsize=13, pad=15, fontweight='bold')
plt.xlabel("Escenario de la Cadena de Suministro", fontsize=11)
plt.ylabel("Margen de Ganancia del Pedido (Ratio)", fontsize=11)
plt.xticks([0, 1], ["A Tiempo / Anticipado (Sin Riesgo)", "Con Riesgo de Entrega Tardía"])
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("grafico1_boxplot_margen.png", bbox_inches='tight', dpi=300)
plt.show()


### 📝 Interpretación Económica del Gráfico 1 (Análisis Descriptivo)

El diagrama de caja (Boxplot) permite contrastar la distribución del **margen de ganancia operativa** entre las órdenes clasificadas `Sin Riesgo` y aquellas `Con Riesgo de Entrega Tardía`.

En ambos escenarios logísticos, la **mediana del margen** se sitúa en torno al 12%, y la caja intercuartílica (que concentra el 50% central de los pedidos) muestra un comportamiento alineado entre ambos grupos. No obstante, al observar la distribución de los valores atípicos (*outliers*), se evidencia la presencia de observaciones con márgenes fuertemente negativos en ambas categorías, alcanzando valores de pérdida que superan el -100%.

Dado que la base de datos no contiene variables que identifiquen factores causales directos (como registros de devoluciones, reclamos o costos específicos de transporte), el análisis se limita a constatar la existencia y dispersión de estos valores atípicos sin inferir causas operativas no medidas.

### Gráfico 2: Análisis de Brecha de Rentabilidad por Segmento de Cliente
**Objetivo metodológico:** Descomponer el agregado global mediante un gráfico de barras agrupadas (*Grouped Bar Chart*) para identificar si el margen de ganancia operativa promedio varía según el perfil comercial del comprador (`Consumer`, `Corporate`, `Home Office`) al cruzarse con el riesgo logístico.


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    x="segmento_cliente",
    y="margen_ganancia_pedido",
    hue="riesgo_entrega_tardia",
    data=df_pedidos_analisis,
    palette="Blues_d",
    errorbar=None
)

plt.title("Margen de Ganancia Operativa Promedio por Segmento y Estado Logístico", fontsize=13, pad=15, fontweight='bold')
plt.xlabel("Segmento Comercial de Clientes", fontsize=11)
plt.ylabel("Margen de Ganancia Promedio (Ratio)", fontsize=11)
plt.legend(title="Logística del Pedido", labels=["A Tiempo", "Con Riesgo de Retraso"])

plt.tight_layout()
plt.savefig("grafico2_barras_segmentos.png", bbox_inches='tight', dpi=300)
plt.show()


### 📝 Interpretación Económica del Gráfico 2 (Análisis Descriptivo)

El gráfico de barras agrupadas muestra que el **margen de ganancia promedio** oscila entre el 11.5% y el 12.5% a través de los tres segmentos de clientes (`Consumer`, `Corporate` y `Home Office`). Al comparar dentro de cada segmento las órdenes con y sin riesgo de entrega tardía, las barras presentan alturas prácticamente equivalentes.

En términos descriptivos, el promedio muestral de la rentabilidad porcentual no muestra diferencias sustanciales asociadas a la alerta de riesgo de retraso en ninguno de los canales comerciales. Conviene remarcar que, al tratarse de un gráfico basado en promedios, la métrica describe la tendencia central de cada grupo y encubre la dispersión individual y los valores atípicos negativos identificados en el gráfico de cajas.

### Gráfico 3: Matriz de Calor Territorial de Rendimiento Económico e Ineficiencia
**Objetivo metodológico:** Emplear una matriz de calor (*Matrix Heatmap*) de doble entrada para aislar el comportamiento geográfico cruzando las 10 regiones comerciales líderes en volumen con el estado de riesgo logístico.


In [ ]:
plt.figure(figsize=(12, 8))

top_10_regiones = df_pedidos_analisis["region"].value_counts().head(10).index
df_filtrado_regiones = df_pedidos_analisis[df_pedidos_analisis["region"].isin(top_10_regiones)]

matriz_pivote = df_filtrado_regiones.pivot_table(
    index="region",
    columns="riesgo_entrega_tardia",
    values="margen_ganancia_pedido",
    aggfunc="mean"
)

sns.heatmap(
    matriz_pivote,
    annot=True,
    cmap="YlGnBu",
    fmt=".2f",
    cbar_kws={'label': 'Margen de Ganancia Operativa Promedio'}
)

plt.title("Matriz de Interacción: Margen Operativo por Región Comercial y Riesgo", fontsize=13, pad=15, fontweight='bold')
plt.xlabel("Riesgo de Entrega Tardía", fontsize=11)
plt.ylabel("Región Comercial del Pedido (Top 10)", fontsize=11)
plt.xticks([0.5, 1.5], ["Sin Riesgo", "Con Riesgo"])

plt.tight_layout()
plt.savefig("grafico3_heatmap_regiones.png", bbox_inches='tight', dpi=300)
plt.show()


### 📝 Interpretación Económica del Gráfico 3 (Análisis Descriptivo)

La matriz de calor ilustra los promedios del margen de ganancia cruzando las 10 principales regiones comerciales con la condición de riesgo de entrega tardía. Se observa que el margen promedio registra variaciones según la **región geográfica** de destino (mostrando tonalidades diferenciadas entre regiones), mientras que las diferencias entre la columna sin riesgo y con riesgo dentro de un mismo territorio resultan marginales (generalmente de fracciones de punto porcentual).

Desde una lectura descriptiva, el margen operativo promedio difiere entre mercados geográficos, pero se mantiene estable al comparar envíos a tiempo frente a envíos con riesgo dentro de cada región. Como consideración metodológica resguardada por la cátedra, cada celda expresa un valor promedio territorial, por lo que el mapa de calor sintetiza el comportamiento medio regional sin detallar la variabilidad interna de los pedidos en cada destino.

## 🤖 Registro y Declaración de Uso de IA

Se deja constancia de que la Sección 4 contó con la asistencia de un modelo de lenguaje de Inteligencia Artificial como apoyo metodológico y técnico.

* **Rol del Agente:** Analista de Datos Senior y Asesor Financiero.
* **Instrucción (Prompt) utilizada:** «En base al DataFrame consolidado por pedido único 'df_pedidos_analisis' de la Sección 2, generá el código de producción en Python para tres gráficos de tipos distintos (Boxplot, Barras agrupadas y Heatmap) cruzando el riesgo logístico (Late_delivery_risk) con el indicador económico real de la tabla ('margen_ganancia_pedido'). Asegurá títulos académicos formales, rotulado correcto de ambos ejes, leyendas explícitas y la exportación de las imágenes en alta resolución con plt.savefig(). Generá también la interpretación teórica e instrumental en celdas de texto Markdown resguardando el enfoque descriptivo y evitando afirmaciones causales no respaldadas por los datos.»
